# **pi-H3 DWD (T2I T2V I2V)**
- Github repo: https://github.com/Lakonik/ComfyUI-piFlow

- Notebook source: https://github.com/Isi-dev/Google-Colab_Notebooks

- [Usage Guide](https://penioj.blogspot.com/2026/11/fast-video-generation-on-google-colab.html)



In [ ]:
# @title 💥 1. Prepare Environment & Download pi-H3 DWD Models {"single-column":true}
import os
import sys
import shutil
import subprocess
import re
import time
import gc
import importlib.util
from pathlib import Path
from IPython.display import clear_output, display, HTML, Image as IPImage

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# 1. Detect GPU Architecture (SM75=T4, SM80=A100, SM89=L4)
import torch
major, minor = torch.cuda.get_device_capability(0)
SM_ARCH = f"SM{major}{minor}"
GPU_NAME = torch.cuda.get_device_name(0)
print(f"🖥️ Detected GPU: {GPU_NAME} ({SM_ARCH})")

# 2. Clone ComfyUI (>= v0.45.0 required for pi-H3) & Custom Nodes
%cd /content
if not os.path.exists("ComfyUI"):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git

%cd /content/ComfyUI/custom_nodes
if not os.path.exists("ComfyUI-piFlow"):
    !git clone https://github.com/Lakonik/ComfyUI-piFlow.git
if not os.path.exists("ComfyUI_GGUF"):
    !git clone --branch leejet_27_09_2026 https://github.com/Isi-dev/ComfyUI_GGUF.git
if not os.path.exists("ComfyUI_VideoHelperSuite"):
    !git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite ComfyUI_VideoHelperSuite
if not os.path.exists("ComfyUI_KJNodes"):
    !git clone https://github.com/kijai/ComfyUI-KJNodes ComfyUI_KJNodes
if not os.path.exists("MiniMax_H3_Combined_Image_And_Reference_to_Video"):
    !git clone https://github.com/Isi-dev/MiniMax_H3_Combined_Image_And_Reference_to_Video.git

# Install Sol-Attn Triton on L4 (SM89) and A100 (SM80)
if SM_ARCH in ("SM80", "SM89") and not os.path.exists("ComfyUI-SolAttn_triton"):
    !git clone https://github.com/kijai/ComfyUI-SolAttn_triton.git

%cd /content/ComfyUI
!pip install -q -r requirements.txt
!pip install -q -r custom_nodes/ComfyUI-piFlow/requirements.txt
!pip install -q -r custom_nodes/ComfyUI_VideoHelperSuite/requirements.txt
!pip install -q -r custom_nodes/ComfyUI_KJNodes/requirements.txt
if os.path.exists("custom_nodes/ComfyUI-SolAttn_triton/requirements.txt"):
    !pip install -q -r custom_nodes/ComfyUI-SolAttn_triton/requirements.txt
!pip install -q gguf huggingface_hub hf_transfer sageattention triton imageio imageio-ffmpeg opencv-python
!apt -y install -qq aria2 ffmpeg

clear_output()
print(f"✅ Repositories & requirements installed for {GPU_NAME} ({SM_ARCH}).")

# @markdown ---
# @markdown ### **Authentication**
hf_token = "" # @param {type:"string"}

# @markdown ---
# @markdown ### **Core pi-H3 DWD Model Selection**
# @markdown *Use INT8 ConvRot for T4/L4/A100 speed & VRAM efficiency.*
diffusion_model_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors" # @param {type:"string"}
dwd_adapter_url = "https://huggingface.co/Lakonik/pi-H3/resolve/main/pi_h3_4step_dwd_adapter_int8_convrot.safetensors" # @param {type:"string"}
text_encoder_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors" # @param {type:"string"}
audio_vae_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors" # @param {type:"string"}

# @markdown *Video VAE is ONLY needed if you plan to run Image-to-Video (first/last frame encoding). DWD skips video VAE decoding completely.*
download_video_vae_for_i2v = True # @param {type:"boolean"}
video_vae_url = "https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors" # @param {type:"string"}

from huggingface_hub import hf_hub_download

def download_hf_file(repo_id, repo_relative_path, local_destination, token=None):
    if not repo_relative_path or repo_relative_path == "None":
        return None
    os.makedirs(local_destination, exist_ok=True)
    filename = os.path.basename(repo_relative_path)
    destination = os.path.join(local_destination, filename)
    if os.path.isfile(destination):
        print(f"⏭️ Already exists: {destination}")
        return filename
    print(f"⬇️ Downloading HF File: {repo_id}/{repo_relative_path}")
    try:
        downloaded_path = hf_hub_download(
            repo_id=repo_id,
            filename=repo_relative_path,
            token=token if token else None
        )
        shutil.copy2(downloaded_path, destination)
        try:
            os.remove(downloaded_path)
        except Exception:
            pass
        print(f"✅ Downloaded: {destination}")
        return filename
    except Exception as e:
        print(f"❌ Download failed for {repo_relative_path}: {e}")
        return None

def download_generic_url(url, dest_dir, token_hf=""):
    if not url or not url.strip():
        return None
    url = url.strip()
    hf_match = re.search(r'huggingface\.co/([^/]+/[^/]+)/(?:resolve|blob)/[^/]+/(.+)', url)
    if hf_match:
        return download_hf_file(hf_match.group(1), hf_match.group(2), dest_dir, token_hf)
    filename = url.split('/')[-1].split('?')[0]
    os.makedirs(dest_dir, exist_ok=True)
    cmd = ['aria2c', '--console-log-level=error', '-c', '-x', '16', '-s', '16', '-k', '1M', '-d', dest_dir, '-o', filename, url]
    subprocess.run(cmd, check=False)
    return filename

token_str = hf_token.strip()
print("--- Downloading pi-H3 DWD Models ---")
unet_file = download_generic_url(diffusion_model_url, "models/diffusion_models", token_str)
dwd_adapter_file = download_generic_url(dwd_adapter_url, "models/loras", token_str)
text_enc_file = download_generic_url(text_encoder_url, "models/text_encoders", token_str)
audio_vae_file = download_generic_url(audio_vae_url, "models/vae", token_str)
video_vae_file = download_generic_url(video_vae_url, "models/vae", token_str) if download_video_vae_for_i2v else None

# Clean HF cache to preserve Colab disk space
!rm -rf /root/.cache/huggingface/*

# 3. Initialize Pure-Python ComfyUI & ComfyUI-piFlow Node Classes
sys.path.insert(0, '/content/ComfyUI')
from comfy import model_management
from nodes import UNETLoader, CLIPLoader, VAELoader, LoadImage, SaveImage
from comfy_extras.nodes_minimax_h3 import MiniMaxH3ImageToVideo
from comfy_extras.nodes_custom_sampler import BasicGuider, RandomNoise, SamplerCustomAdvanced
from comfy_extras.nodes_video import CreateVideo
from comfy_extras.nodes_audio import VAEDecodeAudio

try:
    from custom_nodes.MiniMax_H3_Combined_Image_And_Reference_to_Video.minimax_h3_combined_image_and_reference import (
        MiniMaxH3CombinedImageAndReferenceToVideo,
    )
except ImportError:
    from custom_nodes.MiniMax_H3_Combined_Image_And_Reference_to_Video import (
        MiniMaxH3CombinedImageAndReferenceToVideo,
    )

print("✅ Loaded MiniMaxH3CombinedImageAndReferenceToVideo")

# Dynamically load ComfyUI-piFlow NODE_CLASS_MAPPINGS (works even if filenames change)
piflow_init = "/content/ComfyUI/custom_nodes/ComfyUI-piFlow/__init__.py"
spec = importlib.util.spec_from_file_location("comfyui_piflow", piflow_init)
piflow_mod = importlib.util.module_from_spec(spec)
sys.modules["comfyui_piflow"] = piflow_mod
spec.loader.exec_module(piflow_mod)
PIFLOW_NODES = getattr(piflow_mod, "NODE_CLASS_MAPPINGS", {})

print("\n🔌 Registered ComfyUI-piFlow Nodes:")
for k in PIFLOW_NODES.keys():
    print(f"  - {k}")

def invoke_node(node_cls, **kwargs):
    """Universal pure-Python invoker for both ComfyUI V1 (FUNCTION) and V3 (execute) nodes."""
    inst = node_cls()
    fn_name = getattr(node_cls, "FUNCTION", "execute")
    fn = getattr(inst, fn_name)
    res = fn(**kwargs)
    if type(res).__name__ == 'NodeOutput' or hasattr(res, "__iter__"):
        return list(res)
    return [res]

def find_piflow_node(keywords):
    """Finds a node class in PIFLOW_NODES matching all given substrings (case-insensitive)."""
    for name, cls in PIFLOW_NODES.items():
        low = name.lower()
        if all(kw.lower() in low for kw in keywords):
            return name, cls
    raise KeyError(f"Could not find ComfyUI-piFlow node matching {keywords}. Available: {list(PIFLOW_NODES.keys())}")

def display_video(video_path):
    from base64 import b64encode
    if not os.path.exists(video_path):
        print(f"File not found: {video_path}")
        return
    video_data = open(video_path, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(video_data).decode()
    display(HTML(f'''
    <video width=640 height=360 controls autoplay loop>
        <source src="{data_url}" type="video/mp4">
    </video>
    '''))
clear_output()
print("\n✅ Environment, models, and pure-Python pi-H3 DWD nodes are ready.")

In [ ]:
# @markdown # 📥 2. Upload Images & View Media Bank
import os
import math
from google.colab import files
from PIL import Image
import matplotlib.pyplot as plt

BANK_DIR = '/content/ComfyUI/input'
os.makedirs(BANK_DIR, exist_ok=True)
os.chdir(BANK_DIR)

print(f"Upload reference images to {BANK_DIR} (optional — skip if running Text-to-Video):")
uploaded = files.upload()
os.chdir('/content/')

all_files = os.listdir(BANK_DIR)
available_images = sorted([f for f in all_files if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))])

if available_images:
    print(f"\n--- 🖼️ Image Bank Inventory ({len(available_images)} images) ---")
    cols = 4
    rows = math.ceil(len(available_images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    axes = [axes] if (rows == 1 and cols == 1) else axes.flatten()

    for i, img_name in enumerate(available_images):
        img = Image.open(os.path.join(BANK_DIR, img_name))
        img.thumbnail((512, 512))
        axes[i].imshow(img)
        axes[i].set_title(f"Image [{i+1}]\n{img_name[:18]}", fontsize=11, fontweight='bold')
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("\nNo images in the bank yet.")

In [ ]:
# @markdown # 🎯 3. Select Optional First / Middle / Last Frames
# @markdown Leave any slot empty to skip it. You can use 0, 1, 2, or all 3 frames.
import os

BANK_DIR = '/content/ComfyUI/input'
os.makedirs(BANK_DIR, exist_ok=True)
available_images = sorted([f for f in os.listdir(BANK_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))])

first_frame_number = "1" # @param {type:"string"}
middle_frame_number = "2" # @param {type:"string"}
last_frame_number = "3" # @param {type:"string"}

def get_file_path(num_str, file_list):
    if num_str.strip().isdigit():
        idx = int(num_str.strip()) - 1
        if 0 <= idx < len(file_list):
            return os.path.join(BANK_DIR, file_list[idx])
        print(f"⚠️ Warning: Image [{num_str.strip()}] is out of range (1..{len(file_list)}).")
    return None

first_frame_path = get_file_path(first_frame_number, available_images)
middle_frame_path = get_file_path(middle_frame_number, available_images)
last_frame_path = get_file_path(last_frame_number, available_images)

print("--- Selected Target Keyframes ---")
print(f"First Frame  (t=0):    {os.path.basename(first_frame_path) if first_frame_path else 'None (skipped)'}")
print(f"Middle Frame (t=mid):  {os.path.basename(middle_frame_path) if middle_frame_path else 'None (skipped)'}")
print(f"Last Frame   (t=end):  {os.path.basename(last_frame_path) if last_frame_path else 'None (skipped)'}")

In [5]:
# @title 💥 4. Generate Video / Image {"single-column":true}
import os
import gc
import time
import random
import inspect
import torch
import folder_paths
from comfy import model_management
from comfy_api.latest import Types
from comfy_extras.nodes_custom_sampler import (
    BasicGuider,
    RandomNoise,
    KSamplerSelect,
    SamplerCustomAdvanced,
)
from nodes import SaveImage
from IPython.display import clear_output, display, Image as IPImage

# @markdown ### Prompt & Output Type
output_type = "Video" # @param ["Video", "Single Image (1 frame)"]
prompt = "A cinematic medium shot of a detective walking down a rain-soaked neon street at night, looking over his shoulder as distant thunder rumbles and footsteps splash in puddles." # @param {type:"string"}

# @markdown ### Active Keyframes (each optional; requires corresponding slot set in Cell 3)
use_first_frame = False # @param {type:"boolean"}
use_middle_frame = False # @param {type:"boolean"}
use_last_frame = False # @param {type:"boolean"}

# @markdown ### Canvas & Resolution
resolution_source = "use provided width and height values" # @param ["use provided width and height values", "use width and height of source image first", "resize source (keep aspect ratio, fit mode: total pixels)", "resize source (keep aspect ratio, fit mode: shortest edge)", "resize source (keep aspect ratio, fit mode: longest edge)", "resize source (fit mode: stretch)"]
width = 1024 # @param {type:"integer"}
height = 576 # @param {type:"integer"}
duration_seconds = 3 # @param {type:"number"}

# @markdown ### 4-Step pi-H3 DWD Sampling
steps = 4 # @param [2, 4] {type:"raw"}
seed = 0 # @param {type:"integer"}
video_shift = 12.0 # @param {type:"number"}
audio_shift = 3.0 # @param {type:"number"}

if seed == 0:
    seed = random.randint(1, 100000000000000)

width = max(32, round(width / 32) * 32)
height = max(32, round(height / 32) * 32)

if output_type == "Single Image (1 frame)":
    length = 1
else:
    raw_frames = max(5, int(round(duration_seconds * 24)))
    length = raw_frames + (5 * (raw_frames % 17)) % 17

def build_kwargs_with_defaults(cls, overrides):
    """Populates V1 ComfyUI node arguments using overrides first, then INPUT_TYPES() defaults."""
    fn_name = getattr(cls, "FUNCTION", "execute")
    sig = inspect.signature(getattr(cls(), fn_name))
    it = cls.INPUT_TYPES() if hasattr(cls, "INPUT_TYPES") else {}
    all_specs = {**it.get("required", {}), **it.get("optional", {})}

    kwargs = {}
    for pname, param in sig.parameters.items():
        if pname in overrides and overrides[pname] is not None:
            kwargs[pname] = overrides[pname]
        elif pname in all_specs:
            spec = all_specs[pname]
            if isinstance(spec, (list, tuple)) and len(spec) > 1 and isinstance(spec[1], dict) and "default" in spec[1]:
                kwargs[pname] = spec[1]["default"]
            elif isinstance(spec, (list, tuple)) and len(spec) > 0 and isinstance(spec[0], list) and spec[0]:
                kwargs[pname] = spec[0][0]
        elif param.default is not inspect.Parameter.empty:
            kwargs[pname] = param.default
    return kwargs

def _load_optional_frame(enabled, path_var_name, load_img_node):
    if not enabled:
        return None
    path = globals().get(path_var_name)
    if path and os.path.exists(path):
        return load_img_node.load_image(os.path.basename(path))[0]
    return None

def generate_pi_h3_dwd():
    model = clip = vae_video = vae_audio = None
    cond_out = latent_out = sampled_out = decoded_images = decoded_audio = None

    try:
        with torch.inference_mode():
            # ---------------------------------------------------------
            # STAGE 1: Text + Optional First/Middle/Last Frame Encoding
            # ---------------------------------------------------------
            load_img = LoadImage()
            first_tensor = _load_optional_frame(use_first_frame, "first_frame_path", load_img)
            middle_tensor = _load_optional_frame(use_middle_frame, "middle_frame_path", load_img)
            last_tensor = _load_optional_frame(use_last_frame, "last_frame_path", load_img)

            active_roles = [
                role for role, t in [
                    ("First", first_tensor),
                    ("Middle", middle_tensor),
                    ("Last", last_tensor),
                ] if t is not None
            ]
            mode_label = "+".join(active_roles) if active_roles else "Text-Only (T2V/T2I)"
            print(f"Active Seed: {seed} | Frames: {length} | Steps: {steps} | Conditioning: {mode_label}")

            print("1/4 Loading Qwen3-VL 32B Text Encoder...")
            clip = CLIPLoader().load_clip(text_enc_file, type="minimax", device="default")[0]

            # Load Video VAE ONLY if at least one keyframe is active (used for encoding only)
            if active_roles:
                if not video_vae_file:
                    raise RuntimeError(
                        "One or more keyframes are active, which requires the Video VAE for frame encoding. "
                        "Enable download_video_vae_for_i2v in Cell 1."
                    )
                print("Loading Video VAE (encoder only for active keyframes)...")
                vae_video = VAELoader().load_vae(video_vae_file)[0]

            print("Encoding prompt & optional target keyframes...")
            cond_res = MiniMaxH3CombinedImageAndReferenceToVideo.execute(
                clip=clip,
                vae=vae_video,
                audio_vae=None,  # Not needed when no ref_audios / ref_video_audios are supplied
                prompt=prompt,
                resolution_source=resolution_source,
                width=width,
                height=height,
                length=length,
                first_frame=first_tensor,
                middle_frame=middle_tensor,
                last_frame=last_tensor,
            )
            cond_out, latent_out = list(cond_res)[:2]

            # Immediately purge 32B Text Encoder & Video VAE from VRAM
            clip = vae_video = first_tensor = middle_tensor = last_tensor = None
            model_management.unload_all_models()
            gc.collect()
            torch.cuda.empty_cache()

            # ---------------------------------------------------------
            # STAGE 2: Load Base Model via UNETLoader + Load LakonLab Adapter
            # ---------------------------------------------------------
            print(f"2/4 Loading Base Model ({unet_file})...")
            if "gguf" in unet_file.lower():
                from custom_nodes.ComfyUI_GGUF.nodes import UnetLoaderGGUF
                model = UnetLoaderGGUF().load_unet(unet_file)[0]
            else:
                model = UNETLoader().load_unet(unet_file, weight_dtype="default")[0]

            print(f"Applying {steps}-Step DWD Adapter ({dwd_adapter_file}) via Load LakonLab Adapter...")
            AdapterCls = PIFLOW_NODES["Load LakonLab Adapter"]
            model = invoke_node(
                AdapterCls,
                model=model,
                adapter_name=dwd_adapter_file,
                adapter_strength=1.0
            )[0]

            # Apply ModelSamplingMiniMaxH3 (video_shift=12, audio_shift=3)
            ms_cls = None
            for mod_name in ("comfy_extras.nodes_minimax_h3", "comfy_extras.nodes_model_advanced"):
                try:
                    mod = __import__(mod_name, fromlist=["ModelSamplingMiniMaxH3"])
                    ms_cls = getattr(mod, "ModelSamplingMiniMaxH3", None)
                    if ms_cls is not None:
                        break
                except Exception:
                    pass
            if ms_cls is not None:
                ms_kwargs = build_kwargs_with_defaults(ms_cls, {
                    "model": model,
                    "video_shift": video_shift,
                    "audio_shift": audio_shift,
                })
                model = invoke_node(ms_cls, **ms_kwargs)[0]
                print(f"Applied ModelSamplingMiniMaxH3 (video_shift={video_shift}, audio_shift={audio_shift}).")

            # ---------------------------------------------------------
            # STAGE 3: pi-Flow Scheduler + Euler + SamplerCustomAdvanced + DWD Decode
            # ---------------------------------------------------------
            print(f"3/4 Running {steps}-step pi-H3 DWD Sampling (No Video VAE Decoder)...")
            guider = BasicGuider().get_guider(model, cond_out)[0]
            noise = RandomNoise().get_noise(seed)[0]
            sampler = KSamplerSelect().get_sampler("euler")[0]

            PiFlowSchedCls = PIFLOW_NODES["pi-Flow Scheduler"]
            sched_kwargs = build_kwargs_with_defaults(PiFlowSchedCls, {
                "model": model,
                "steps": steps,
                "max_raw_t": 0.96 if steps == 4 else 0.93,
            })
            sigmas = invoke_node(PiFlowSchedCls, **sched_kwargs)[0]

            sampled_out = SamplerCustomAdvanced().sample(
                noise=noise,
                guider=guider,
                sampler=sampler,
                sigmas=sigmas,
                latent_image=latent_out
            )[0]

            print("Decoding RGB frames directly via DWD Decode (no Video VAE)...")
            DWDDecodeCls = PIFLOW_NODES["DWD Decode"]
            decoded_images = invoke_node(DWDDecodeCls, samples=sampled_out)[0]

            # Free Diffusion Model from VRAM before Audio VAE / Saving
            model = guider = sampler = None
            model_management.unload_all_models()
            gc.collect()
            torch.cuda.empty_cache()

            # ---------------------------------------------------------
            # STAGE 4: Save Image (length=1) OR Decode Audio & Export MP4
            # ---------------------------------------------------------
            timestamp = time.strftime("%Y%m%d_%H%M%S")
            output_prefix = f"piH3_DWD_{steps}step_{timestamp}"

            if length == 1:
                print("4/4 Saving Single Image Output...")
                SaveImage().save_images(images=decoded_images, filename_prefix=output_prefix)
                out_dir = folder_paths.get_output_directory()
                matches = sorted([
                    os.path.join(out_dir, f) for f in os.listdir(out_dir)
                    if f.startswith(output_prefix) and f.endswith((".png", ".jpg"))
                ])
                clear_output()
                if matches:
                    print(f"✅ Saved: {matches[-1]}")
                    display(IPImage(filename=matches[-1]))
                else:
                    print("❌ Image saved, but file could not be located.")
            else:
                print("4/4 Decoding Audio & Exporting MP4...")
                vae_audio = VAELoader().load_vae(audio_vae_file)[0]
                decoded_audio = VAEDecodeAudio().decode(vae_audio, sampled_out)[0]

                vae_audio = None
                gc.collect()
                torch.cuda.empty_cache()

                created_video = invoke_node(
                    CreateVideo,
                    images=decoded_images,
                    fps=24.0,
                    audio=decoded_audio,
                    bit_depth=8,
                    color_space="sRGB",
                    codec="none"
                )[0]

                w_vid, h_vid = created_video.get_dimensions()
                full_output_folder, filename, counter, _, _ = folder_paths.get_save_image_path(
                    output_prefix,
                    folder_paths.get_output_directory(),
                    w_vid,
                    h_vid
                )
                ext = Types.VideoContainer.get_extension("auto")
                final_output_path = os.path.join(full_output_folder, f"{filename}_{counter:05}_.{ext}")

                created_video.save_to(
                    final_output_path,
                    format=Types.VideoContainer("auto"),
                    codec="auto",
                    metadata=None
                )

                clear_output()
                print(f"✅ Saved ({mode_label}): {final_output_path}")
                display_video(final_output_path)

    finally:
        model = clip = vae_video = vae_audio = cond_out = latent_out = sampled_out = decoded_images = decoded_audio = None
        model_management.unload_all_models()
        gc.collect()
        torch.cuda.empty_cache()

generate_pi_h3_dwd()

✅ Saved (Text-Only (T2V/T2I)): /content/ComfyUI/output/piH3_DWD_4step_20261125_204323_00001_.mp4
